<a href="https://colab.research.google.com/github/netsetos/agentic-ai-weekend-gcp-learners/blob/main/module-10-tuning-and-evaluation/lesson-10.3-batch-routing/notebooks/GCP_Capstone_10.3_BatchRouting.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 10.3 Batch API + Model Routing — The Same Request as a Job, and the Router Finally Called
**Netsetos GenAI Engineering — GCP Capstone** · Module 10 · rebuilt on the live lane, 9 September 2026

Two discounts the lane had not claimed. Batch: the dataset build from 10.1 is the one workload nobody waits on, so it is submitted as a job - the same request, half price, idempotent, with a nightly function and a failure path that is just the online call. Routing: `router.py` has been in the API image since Module 3 with nothing calling it; here it classifies the 64 golden questions, the budget breaker reads the month's counter, and `ROUTING=on` shows up where everything on the lane shows up - in the usage row, priced at the tier's rate.


## Setup


In [ ]:
!pip install -q google-genai==2.22.0 google-cloud-storage==3.13.1 google-cloud-firestore==2.30.0 pandas==2.3.3 requests==2.34.2

from google.colab import auth
auth.authenticate_user()

PROJECT_ID = "documind-ai-YOUR-ID"   # CHANGE THIS: the project the lane runs in (make up, lesson 4.8)
REGION     = "us-central1"
TENANT     = "acme"
KIT        = "/content/agentic-ai-weekend-gcp"   # the kit: deploy/shared is the tool layer every lesson on the lane imports
BRANCH     = "feat/lesson-4.8-live-evals"        # the demo branch; main is behind it

import os, subprocess, sys
import google.auth
from google.auth.transport.requests import AuthorizedSession
from google import genai
from google.genai import types

if not os.path.isdir(KIT):
    subprocess.run(["git", "clone", "--depth", "1", "-q", "-b", BRANCH,
                    "https://github.com/netsetos/agentic-ai-weekend-gcp", KIT], check=True)
sys.path.insert(0, f"{KIT}/deploy")                   # `from shared import ...` - the same layer every service imports

# The lane's URLs are deterministic: service name + project NUMBER (eventarc.tf builds them the same way).
creds, _ = google.auth.default()
NUMBER = AuthorizedSession(creds).get(
    f"https://cloudresourcemanager.googleapis.com/v1/projects/{PROJECT_ID}").json()["projectNumber"]
API_URL       = f"https://documind-api-{NUMBER}.{REGION}.run.app"
UPLOAD_BUCKET = f"{PROJECT_ID}-uploads"     # storage.tf: the bucket eventarc.tf watches - the corpus, media included
MEDIA_BUCKET  = f"{PROJECT_ID}-media"       # storage.tf: generated assets, 30-day lifecycle (a cache, not a record)
DATASETS      = f"{PROJECT_ID}-datasets"    # storage.tf (Module 10): the frozen tuning dataset and 10.5's GGUF
os.environ.update({
    "GOOGLE_CLOUD_PROJECT": PROJECT_ID,
    "GOOGLE_CLOUD_LOCATION": "global",              # Gemini 3.x generation is served from the global endpoint
    "GOOGLE_GENAI_USE_VERTEXAI": "TRUE",
    "DOCUMIND_PROFILE": "gcp",
    "RAG_API_URL": API_URL,
    "RAG_TIMEOUT_S": "90",                          # 7.2's finding: a cold API takes longer than the default 20 s
    # A notebook has no metadata server to be anyone with: the kit mints its ID tokens AS this roster
    # member (7.1). On Cloud Run the service's own account is the identity and nothing is set.
    "DOCUMIND_IMPERSONATE_SA": f"documind-ui-sa@{PROJECT_ID}.iam.gserviceaccount.com",
})
MEMBER_SA   = os.environ["DOCUMIND_IMPERSONATE_SA"]
OUTSIDER_SA = f"documind-outsider-sa@{PROJECT_ID}.iam.gserviceaccount.com"   # IAM admits it, no roster does (4.8, 7.2)

from shared import documind_tools    # THE one retrieve(). Imported, never pasted - the contract gate fails a paste.
gen = genai.Client(enterprise=True, project=PROJECT_ID, location="global")   # every generate_content in this lesson
ROWS       = 40      # chunks the batch cell submits; make trainset --batch submits 300
BUDGET_USD = 100.0   # the month's cap the revision carries (12.2 DEPLOY: BUDGET_USD)

print("kit:", KIT, "| API:", API_URL, "| datasets:", f"gs://{DATASETS}/sft/")


## Cell 1: The API and its usage rows


In [ ]:
import json, requests, time, subprocess, datetime
from google.cloud import storage

# THE API, CALLED THE WAY THE UI CALLS IT: one ID token per request, minted AS the roster member,
# audience = the API (7.3's hour-long fuse never arms). The kit mints it (documind_tools._id_token).
def api(path: str, body: dict | None = None, base: str | None = None, timeout: int = 120) -> tuple[int, dict | str]:
    """POST one API route (or a candidate revision's, with base=) as documind-ui-sa. Returns (status, json-or-text)."""
    url = (base or API_URL).rstrip("/")
    r = requests.post(f"{url}{path}", json=body,
                      headers={"Authorization": f"Bearer {documind_tools._id_token(url)}"}, timeout=timeout)
    try:
        return r.status_code, r.json()
    except ValueError:
        return r.status_code, r.text[:400]

# The usage rows the API logs - the ONE shape every observability consumer reads (12.3, tenant_daily). On
# the lean lane they live in Cloud Logging; this reads the last few for a surface, newest first.
def usage_rows(minutes: int = 15, limit: int = 20, event: str = "query") -> list[dict]:
    since = (datetime.datetime.now(datetime.timezone.utc) - datetime.timedelta(minutes=minutes)).strftime("%Y-%m-%dT%H:%M:%SZ")
    r = subprocess.run(["gcloud", "logging", "read",
                        f'resource.type="cloud_run_revision" AND resource.labels.service_name="documind-api" '
                        f'AND jsonPayload.event="{event}" AND timestamp>="{since}"',
                        "--project", PROJECT_ID, "--limit", str(limit), "--format=json"], capture_output=True, text=True)
    try:
        return [e["jsonPayload"] for e in json.loads(r.stdout or "[]")]
    except ValueError:
        return []

gcs = storage.Client(project=PROJECT_ID)

def gcs_text(uri: str) -> str:
    bucket, _, name = uri.removeprefix("gs://").partition("/")
    return gcs.bucket(bucket).blob(name).download_as_text()

sys.path.insert(0, f"{KIT}/deploy/evals")               # the kit's builders and judges: make_trainset, judge, tune, run_eval
sys.path.insert(0, f"{KIT}/deploy/services/rag-api")    # the API's own modules: cache_manager, router, breakers, cost
print("helpers: api(), usage_rows(), gcs_text(); the kit's evals/ and rag-api/ on sys.path")


## Cell 2: The batch workload is the dataset build
The same request `ask_pairs` makes, one line per chunk, submitted through a regional client. Submitted twice, one job.


In [ ]:
import make_trainset as mt

# THE BATCH WORKLOAD IS THE DATASET BUILD. Batch is for work nobody is waiting on - half price, no rate
# limit, a 24-hour promise - and the lane has exactly one such job: the question/answer pairs make_trainset
# writes from the corpus, one call per chunk (10.1). Inline that is ask_pairs(); as a batch it is the SAME
# request, one line per chunk, custom_id = the chunk id, submitted through a REGIONAL client (batch is
# regional on Vertex; generation is global - the two-client rule again).
chunks = mt.sample(mt.load_chunks(TENANT, PROJECT_ID), ROWS)
n = mt.write_batch_requests(chunks, "batch_requests.jsonl")
first = json.loads(open("batch_requests.jsonl", encoding="utf-8").readline())
print(f"{n} request lines. One, abridged:")
print("  custom_id       :", first["custom_id"])
print("  generationConfig:", first["request"]["generationConfig"]["responseMimeType"], "+ the Pair schema")
print("  prompt          :", first["request"]["contents"][0]["parts"][0]["text"][:120].replace("\n", " "), "...")

VERSION = f"colab-{datetime.date.today():%Y%m%d}"
src = f"gs://{DATASETS}/sft/batch/requests_{VERSION}.jsonl"
dest = f"gs://{DATASETS}/sft/batch/out_{VERSION}/"
gcs.bucket(DATASETS).blob(f"sft/batch/requests_{VERSION}.jsonl").upload_from_filename("batch_requests.jsonl")
job = mt.submit_batch(PROJECT_ID, src, dest, f"documind-trainset-{VERSION}")
print("\njob  :", job.name, "|", str(job.state).rsplit(".", 1)[-1])
again = mt.submit_batch(PROJECT_ID, src, dest, f"documind-trainset-{VERSION}")
assert again.name == job.name, "a second submit made a second job: the idempotence a Scheduler retry depends on is missing"
print("submitted twice, one job: a Scheduler retry is not a second bill")


## Cell 3: Collect, when it is done


In [ ]:
# COLLECT, WHEN IT IS DONE. A batch outlives any cell. This reads the output prefix and, if the job has
# written predictions.jsonl, finishes the build the way `make trainset --collect` does: the predictions back
# into rows in the shape ask_pairs returns, then the same golden-exclusion rules and the same PII scan.
out = [b for b in gcs.list_blobs(DATASETS, prefix=f"sft/batch/out_{VERSION}/") if b.name.endswith("predictions.jsonl")]
if out:
    lines = out[0].download_as_text().splitlines()
    rows = mt.collect_batch(lines, {c["chunk_id"]: c for c in chunks})
    parsed = sum(1 for r in rows if r["answerable"])
    print(f"{len(lines)} predictions -> {len(rows)} rows ({len(rows) - parsed} refusals); {len(lines) - parsed} lines yielded no row")
    golden = [json.loads(l) for l in open(f"{KIT}/deploy/evals/golden.jsonl", encoding="utf-8") if l.strip()]
    rows, dropped = mt.exclude_golden(rows, golden)
    rows, dropped_pii = mt.redact(rows)
    print(f"after the two golden rules and the PII scan: {len(rows)} rows (dropped {len(dropped)} + {dropped_pii})")
    print("first:", rows[0]["question"][:100])
else:
    state = str(genai.Client(enterprise=True, project=PROJECT_ID, location=REGION).batches.get(name=job.name).state).rsplit(".", 1)[-1]
    print(f"not finished ({state}). Batch promises 24 hours and usually takes minutes: re-run this cell, or from Cloud Shell later\n"
          f"  gcloud storage cp {dest}*/predictions.jsonl . && python evals/make_trainset.py --project {PROJECT_ID} --collect predictions.jsonl")


## Cell 4: The failure path is the online call


In [ ]:
# THE FAILURE PATH IS THE ONLINE CALL. Batch output is one line per request. A line whose response did not
# parse - the model refused, the JSON was cut, an error object came back - yields no row, and collect_batch
# SKIPS it rather than raising: a build of 300 rows must not die on the 212th. The chunks that produced no
# row are re-asked inline (ask_pairs: the same prompt, the same schema, full price, now). Escalation on the
# lane is "the online call", not a bigger model - the request was never the problem.
fixture = [json.dumps({"custom_id": chunks[0]["chunk_id"], "response": {"candidates": [{"content": {"parts": [{"text": json.dumps(
               {"question": "What does the clause require?", "answer": "Sixty days' notice.", "quote": "sixty days", "unanswerable_question": "Who signed it?"})}]}}]}}),
           json.dumps({"custom_id": chunks[1]["chunk_id"], "response": {"candidates": [{"content": {"parts": [{"text": "{not json"}]}}]}}),
           json.dumps({"custom_id": chunks[2]["chunk_id"], "error": {"code": 429, "message": "quota"}})]
by_id = {c["chunk_id"]: c for c in chunks[:3]}
got = mt.collect_batch(fixture, by_id, refusal_every=0)
missing = [cid for cid in by_id if cid not in {r["chunk_id"] for r in got}]
print("rows from the three-line fixture:", len(got), "| chunks with no row:", len(missing))
assert len(got) == 1 and len(missing) == 2, (got, missing)
retry = mt.ask_pairs(PROJECT_ID, [by_id[c] for c in missing])          # inline, now, full price - two calls
print("re-asked inline:", len(retry), "rows |", retry[0]["question"][:100])


## Cell 5: The nightly job
A 2nd-gen function that submits the day's requests file, and the commands that deploy and schedule it. Printed, not run.


In [ ]:
# THE NIGHTLY JOB. Real files, so the deploy command below is a real command: a 2nd-gen function that
# submits the trainset batch for the day's requests file. Three rules it carries, each learned the hard way:
# ONE CloudEvent argument (the 1st-gen signature does not deploy), the Pub/Sub message is base64 INSIDE the
# envelope, and it returns without polling - a batch outlives any function timeout. Idempotence is the
# whole game for a scheduled job: Scheduler retries on a non-2xx, and two identical batches over 100,000
# documents is a duplicated bill, not a duplicated log line.
import pathlib
pathlib.Path("batch_fn").mkdir(exist_ok=True)
MAIN_PY = '''import base64, json, os
import functions_framework
from google import genai
from google.genai.types import CreateBatchJobConfig


@functions_framework.cloud_event
def launch_nightly_trainset(cloud_event):
    raw = base64.b64decode(cloud_event.data["message"]["data"]).decode()
    payload = json.loads(raw) if raw.strip().startswith("{") else {}
    day = payload.get("partition", "nightly")

    # Batch is REGIONAL on Vertex. This client submits a job; it does not generate.
    client = genai.Client(enterprise=True, project=os.environ["PROJECT_ID"], location="us-central1")
    bucket = os.environ["DATASETS"]
    name = f"documind-trainset-{day}"
    for job in client.batches.list():                       # idempotent, like make_trainset.submit_batch
        if job.display_name == name and str(job.state) not in ("JOB_STATE_FAILED", "JOB_STATE_CANCELLED"):
            print(f"{name} already submitted: {job.state}")
            return
    job = client.batches.create(
        model="gemini-3.6-flash",
        src=f"gs://{bucket}/sft/batch/requests_{day}.jsonl",
        config=CreateBatchJobConfig(dest=f"gs://{bucket}/sft/batch/out_{day}/", display_name=name))
    print(f"submitted {job.name}: {job.state}")             # return, do not poll
'''
(pathlib.Path("batch_fn") / "main.py").write_text(MAIN_PY, encoding="utf-8")
(pathlib.Path("batch_fn") / "requirements.txt").write_text("functions-framework==3.*\ngoogle-genai==2.22.0\n", encoding="utf-8")
import ast
ast.parse(MAIN_PY)
assert "@functions_framework.cloud_event" in MAIN_PY
print("batch_fn/main.py written and parses;", [l for l in MAIN_PY.splitlines() if l.startswith("def ")][0])


In [ ]:
# The deploy and schedule commands. Printed, not run: the account, the topic and the schedule are
# infrastructure, and infrastructure on the lane is Terraform's or Cloud Shell's, never a notebook's.
print(f"""
gcloud iam service-accounts create documind-batch-sa --project={PROJECT_ID}
gcloud projects add-iam-policy-binding {PROJECT_ID} --member=serviceAccount:documind-batch-sa@{PROJECT_ID}.iam.gserviceaccount.com --role=roles/aiplatform.user
gcloud storage buckets add-iam-policy-binding gs://{DATASETS} --member=serviceAccount:documind-batch-sa@{PROJECT_ID}.iam.gserviceaccount.com --role=roles/storage.objectAdmin

gcloud pubsub topics create documind-batch-tick --project={PROJECT_ID}

gcloud functions deploy launch-nightly-trainset \\
  --gen2 --runtime=python312 --region={REGION} --project={PROJECT_ID} \\
  --source=batch_fn --entry-point=launch_nightly_trainset \\
  --trigger-topic=documind-batch-tick \\
  --set-env-vars=PROJECT_ID={PROJECT_ID},DATASETS={DATASETS} \\
  --service-account=documind-batch-sa@{PROJECT_ID}.iam.gserviceaccount.com

# 02:00 IST daily. Name the timezone, or Scheduler runs on UTC and "overnight" lands at 07:30 local.
gcloud scheduler jobs create pubsub documind-nightly-trainset \\
  --location={REGION} --project={PROJECT_ID} --schedule="0 2 * * *" --time-zone="Asia/Kolkata" \\
  --topic=documind-batch-tick --message-body='{{"partition":"nightly"}}'
""")


## Cell 6: The classifier the image carried
`router.classify` over the 64 golden questions: a distribution, not a rule.


In [ ]:
from router import classify, ROUTING_TABLE, CLASSIFIER_MODEL
from collections import Counter

# THE CLASSIFIER THE IMAGE CARRIED. router.py has been in the API image since 3.3 wrote it and 12.6 shipped
# it, and nothing on the request path called it until Module 10. classify() is one short flash-lite call
# with an enum schema and thinking at its floor. Here it runs over the 64 golden questions - the lane's own
# traffic shape - and the result is a distribution, not a rule: which of the lane's questions does the
# classifier think need Pro, and what does that cost?
golden = [json.loads(l) for l in open(f"{KIT}/deploy/evals/golden.jsonl", encoding="utf-8") if l.strip()]
t0 = time.time()
tiers = {g["id"]: classify(g["question"], gen).value for g in golden}
print(f"{len(golden)} questions classified by {CLASSIFIER_MODEL} in {time.time() - t0:.0f}s\n")
print(f"{'shape':10} {'SIMPLE':>7} {'MEDIUM':>7} {'COMPLEX':>8}")
for shape in ("lookup", "join", "refusal", "isolation"):
    c = Counter(tiers[g["id"]] for g in golden if g["shape"] == shape)
    print(f"{shape:10} {c['SIMPLE']:>7} {c['MEDIUM']:>7} {c['COMPLEX']:>8}")
DIST = {k: v / len(golden) for k, v in Counter(tiers.values()).items()}
print("\ndistribution:", {k: f"{v:.0%}" for k, v in sorted(DIST.items())})
for k, v in ROUTING_TABLE.items():
    print(f"  {k.value:8} -> {v['model']:24} thinking {v['thinking_budget']:>5}  max_out {v['max_output_tokens']}")
complex_ids = [i for i, t in tiers.items() if t == "COMPLEX"]
print("\nCOMPLEX:", complex_ids or "none - the lane's questions are lookups and joins; Pro would answer nothing here")


## Cell 7: What the service does when the money runs out
`breakers.choose_model` at three spend levels, and the month's counter it reads on the lean lane.


In [ ]:
from breakers import choose_model, routing_mode, BUDGET_STRICT_PCT, BUDGET_FLOOR_PCT
from budget import spend_pct
from google.cloud import firestore

# WHAT THE SERVICE DOES WHEN THE MONEY RUNS OUT - and where it reads the money from. breakers.py has said
# since 12.6 what happens at 80% (strict routing: nothing dearer than flash) and at 100% (min-instances 0,
# a slower service, not a dead one), and read a spend it had no source for. budget.py is the source on the
# lean lane: one Firestore document per month, incremented by the cost of every answer main.py logs. The
# notebook reads the same document the API reads.
db = firestore.Client(project=PROJECT_ID)
pct = spend_pct(db, BUDGET_USD)
month = datetime.datetime.now(datetime.timezone.utc).strftime("%Y-%m")
usd = (db.collection("budget").document(month).get().to_dict() or {}).get("usd", 0.0)
print(f"budget/{month}: ${usd:.4f} of ${BUDGET_USD:.0f} = {pct:.2f}% -> routing {routing_mode(pct)}")
print(f"\n{'tier':8} {'at 20%':24} {'at 85%':24} {'at 100%'}")
for tier in ("SIMPLE", "MEDIUM", "COMPLEX"):
    print(f"{tier:8} {choose_model(tier, 20):24} {choose_model(tier, 85):24} {choose_model(tier, 100)}")
assert choose_model("COMPLEX", 85) == "gemini-3.6-flash" and choose_model("SIMPLE", 20) == "gemini-3.1-flash-lite"
print(f"\nstrict from {BUDGET_STRICT_PCT}%: Pro is never chosen. The floor at {BUDGET_FLOOR_PCT}% is min-instances 0 (12.6), not an outage.")
print("SPEND_PCT=85 on the revision replays the strict month without spending it: the same reading, overridden.")


## Cell 8: ROUTING=on, on the lane
The revision's setting, three questions, and the usage rows' `model` field.


In [ ]:
# ROUTING=on, ON THE LANE. The switch is an environment variable on the API revision (12.2's DEPLOY carries
# it; `make deploy-services ROUTING=on`). This cell reads the revision's setting, asks three questions of
# three shapes through retrieve(), and reads the usage rows back. With routing on, `model` differs per
# question and each row is priced at ITS model's rate (cost.py); with it off, one model answers everything,
# which is what the lane ran until Module 10. Either way the row says which - the row is the proof.
svc = json.loads(subprocess.run(["gcloud", "run", "services", "describe", "documind-api", "--region", REGION,
                                 "--project", PROJECT_ID, "--format=json"], capture_output=True, text=True).stdout)
env = {e["name"]: e.get("value", "") for e in svc["spec"]["template"]["spec"]["containers"][0].get("env", [])}
print("revision env:", {k: env.get(k) for k in ("GENERATOR_MODEL", "ROUTING", "BUDGET_USD", "SPEND_PCT")})
PROBES = [("lookup ", "What is the per-trip cap on domestic travel reimbursement?"),
          ("compare", "Compare the notice period for a confirmed E3 with the notice the MSA requires to terminate for convenience."),
          ("reason ", "If a confirmed E3 resigns on 1 March and ACME must also give the MSA's termination notice the same day, which period ends first, and by how many days?")]
for label, q in PROBES:
    got = documind_tools.retrieve(q, tenant_id=TENANT, brain="direct")
    print(f"  {label} answerable={got.get('answerable')} citations={len(got.get('citations') or [])}")
time.sleep(20)
rows = usage_rows(minutes=3, limit=3)
for r in rows:
    print(f"  model={str(r.get('model')):26} tokens_in={r.get('tokens_in'):>5} cost_usd={r.get('cost_usd')}")
models = {r.get("model") for r in rows}
if env.get("ROUTING") == "on" and len(models) > 1:
    print("\nrouting is ON: the tier answered, and each row is priced at the tier's rate")
elif env.get("ROUTING") == "on":
    print("\nrouting is ON and the classifier put all three in one tier - the lane's questions are mostly lookups; try a fourth")
else:
    print("\nROUTING is off on this revision: one model answered every question. make deploy-services ROUTING=on turns it on.")


## Cell 9: The price of the distribution


In [ ]:
# THE PRICE OF THE DISTRIBUTION. The calculator is 10.3's, with the lane's own inputs: the tier distribution
# the classifier produced on 64 real questions, and tokens per answer from the usage rows. Two discounts
# stack - the tier (what the question needs) and batch (half price for what nobody waits on) - and the
# baseline is the lane's actual state, everything on gemini-3.6-flash standard. Routing can cost MORE than
# that if the classifier sends enough to Pro; the number says so, and that is the point of computing it.
PRICING = {"gemini-3.1-flash-lite":  {"in_std": 0.25, "in_batch": 0.125, "out_std": 1.50, "out_batch": 0.75},
           "gemini-3.6-flash":       {"in_std": 1.50, "in_batch": 0.75,  "out_std": 7.50, "out_batch": 3.75},
           "gemini-3.1-pro-preview": {"in_std": 2.00, "in_batch": 1.00,  "out_std": 12.00, "out_batch": 6.00}}   # USD per 1M tokens, standard rates (CLAUDE.md)
TIER_MODEL = {"SIMPLE": "gemini-3.1-flash-lite", "MEDIUM": "gemini-3.6-flash", "COMPLEX": "gemini-3.1-pro-preview"}

def calc_cost(model: str, tokens_in: int, tokens_out: int, batch: bool = False) -> float:
    p = PRICING[model]
    return (tokens_in * p["in_batch" if batch else "in_std"] + tokens_out * p["out_batch" if batch else "out_std"]) / 1_000_000

def scenario(n: int, avg_in: float, avg_out: float, dist: dict) -> dict:
    routed = lambda batch: sum(calc_cost(TIER_MODEL[t], int(n * f) * avg_in, int(n * f) * avg_out, batch) for t, f in dist.items())
    base = calc_cost("gemini-3.6-flash", n * avg_in, n * avg_out)
    return {"all_flash_standard": round(base, 2), "routed_standard": round(routed(False), 2), "routed_batch": round(routed(True), 2),
            "saving_vs_flash": f"{(1 - routed(True) / base):+.0%}"}

recent = usage_rows(minutes=60 * 24, limit=300)
avg_in = sum(r.get("tokens_in", 0) for r in recent) / max(len(recent), 1) or 1800
avg_out = sum(r.get("tokens_out", 0) for r in recent) / max(len(recent), 1) or 250
print(f"measured: {len(recent)} answers in the last day, {avg_in:,.0f} tokens in / {avg_out:,.0f} out per answer")
for n in (max(len(recent), 1) * 30, 100_000):
    print(f"  {n:>7,} answers a month:", scenario(n, avg_in, avg_out, DIST))
print("\nthe distribution is the lane's; a corpus of contracts to analyse would route more to Pro and the sign could flip")


## Cell 10: Per-tenant attribution, from the row


In [ ]:
import pandas as pd

# PER-TENANT ATTRIBUTION, FROM THE ROW. Every usage row carries tenant, model and cost_usd; tenant_daily.sql
# groups them on the full profile, and this cell groups the same rows on lean, from the last day. Billing-
# export labels (a label on every generate call, UNNEST in the export) are the other way; the lane does not
# need them, because the row already says who asked, which model answered and what it cost.
df = pd.DataFrame(usage_rows(minutes=60 * 24, limit=500))
if len(df):
    by = df.groupby("tenant").agg(answers=("cost_usd", "size"), usd=("cost_usd", "sum"), tokens_in=("tokens_in", "sum"))
    by["inr"] = (by["usd"] * 85).round(2)
    print(by.sort_values("usd", ascending=False).to_string())
    print("\nby model:")
    print(df.groupby("model")["cost_usd"].agg(answers="size", usd="sum").to_string())
    print("\nby brain (8.7's question, answered from the row):")
    print(df.groupby("brain")["cost_usd"].agg(answers="size", usd="sum").to_string())
else:
    print("no usage rows in the last day - ask a question first (Cell 6 does)")


## Where this goes
- **10.4** judges what the routed tiers answered: the same rows, a second judge.
- **12.6** owns the quota that refuses the 601st request before a function reports it; everything here is reactive.

## ✅ Lesson 10.3 complete
- ✅ The dataset build submitted as a batch job: the same request, half price, one job for two submits
- ✅ Collect and the failure path (the online call), and the nightly function with its commands
- ✅ The kit's router classifying the lane's own questions; the breaker reading the month's counter
- ✅ ROUTING=on read off the usage rows, priced per tier
- ✅ The distribution priced against the lane's real state; attribution by tenant, model and brain from the row
